# GPU Production Workflow — Complete Guide

## CPU에서 검증 완료 → GPU에서 실제 결과를 얻기 위한 전체 가이드

### 전체 흐름 (6단계)

```
┌─────────────────────────────────────────────────────────────────┐
│  ① PINN 학습 (GPU)              ② LightTools 시뮬 (회사 PC)   │
│     20K colloc, 50K epochs          TMM+ASM source 주입         │
│     ~4-8시간                        BM 200 조합 배치            │
│          ↓                          ~1-2일                     │
│     phase_c_final.pt                   ↓                       │
│          ↓                    data/lt_results/sim_*.npz        │
│          └──────────┬──────────────────┘                       │
│                     ↓                                          │
│  ③ L_I Fine-tuning (GPU, lambda_I=0.3, ~2시간)                │
│          ↓                                                     │
│  ④ FNO 재증류 (~10분)                                          │
│          ↓                                                     │
│  ⑤ BoTorch 역설계 (~5분)                                      │
│          ↓                                                     │
│  ⑥ Design Studio UI로 결과 확인                                │
└─────────────────────────────────────────────────────────────────┘

★ ①과 ②는 동시 진행 가능! (GPU 학습하면서 LT 시뮬)
```

### CPU vs GPU 차이

| 항목 | CPU (완료) | GPU (목표) |
|------|-----------|----------|
| Collocation | 10,000 | **20,000** |
| Epochs | 3,000 | **50,000** |
| Model | 64-dim, 3-layer | **128-dim, 4-layer** |
| 학습 시간 | ~4분 | **~4-8시간** |
| Interior CoV | 0.45 | **> 0.5 (목표)** |
| Design sensitivity | 0.00002 | **> 0.01 (목표)** |
| L_I | 없음 | **LT 데이터 사용** |

In [ ]:
import sys
from pathlib import Path

def _find_root():
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / 'pyproject.toml').exists(): return p
        p = p.parent
    raise FileNotFoundError('Cannot find project root')

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project: {PROJECT_ROOT}')

---
# Step 0: GPU 환경 사전 점검

GPU PC에서 이 셀을 먼저 실행하세요.

In [ ]:
# ── 사전 점검 ──
import subprocess, sys
result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'preflight_check.py'),
     '--config', str(PROJECT_ROOT / 'configs' / 'phase_c_full_gpu.yaml')],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:] if result.stderr else 'none')

### 점검 실패 시 조치

| FAIL 항목 | 해결 방법 |
|----------|----------|
| CUDA | `pip install torch --index-url https://download.pytorch.org/whl/cu121` |
| ASM LUT | 아래 Step 0a 실행 |
| packages | `pip install -r requirements.txt` |

In [ ]:
# ── Step 0a: ASM LUT가 없으면 생성 ──
import numpy as np

lut_path = PROJECT_ROOT / 'data' / 'asm_luts' / 'incident_z40.npz'
if lut_path.exists():
    print(f'ASM LUT exists: {lut_path.stat().st_size/1024:.0f} KB')
else:
    print('Generating ASM LUT...')
    from backend.physics.tmm_calculator import GorillaDXTMM
    from backend.physics.asm_propagator import generate_incident_lut
    
    tmm = GorillaDXTMM()
    theta = np.arange(-41, 42, 1.0)
    x = np.linspace(0, 504.0, 4096)
    lut = generate_incident_lut(tmm, theta, x)
    
    lut_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(lut_path, **lut,
             z_inlet=np.float32(40), cg_thick=np.float32(550),
             wavelength_nm=np.float32(520), n_cg=np.float32(1.52))
    print(f'Done: {lut_path}')

---
# Step 1: PINN 학습 (GPU)

## ★ 가장 중요한 단계 (~4-8시간)

### CPU에서 배운 인사이트 (반영됨)
```
- n_colloc: 20,000 (CPU 실험: 10K에서 CoV=1.05, 20K로 안전마진)
- warm_start: 500 epochs (수렴 25-40% 가속)
- curriculum 3-stage: BC→PDE ramp→Full
- 자명해 방지: 충분한 collocation이 핵심 (epoch보다 중요!)
```

### 실행 방법 (2가지)

In [ ]:
# ── 방법 A: 터미널에서 실행 (권장, 백그라운드) ──
print('=== 터미널에서 실행할 명령어 ===')
print()
print('# Option 1: 기본 설정 (권장)')
print('python scripts/train_phase_c.py --config configs/phase_c_full_gpu.yaml')
print()
print('# Option 2: Warm Start 건너뛰기 (Cold Start)')
print('python scripts/train_phase_c.py --config configs/phase_c_full_gpu.yaml --no-warmstart')
print()
print('# Option 3: 백그라운드 실행 (터미널 닫아도 계속)')
print('nohup python scripts/train_phase_c.py --config configs/phase_c_full_gpu.yaml > train.log 2>&1 &')
print()
print('# 학습 로그 모니터링')
print('tail -f experiments/YYYY-MM-DD_phase_c/training.log')

In [ ]:
# ── 방법 B: 이 노트북에서 직접 실행 ──
# 주의: 4-8시간 걸리므로 터미널 실행 권장

RUN_HERE = False  # True로 변경하면 여기서 실행

if RUN_HERE:
    import subprocess
    proc = subprocess.Popen(
        [sys.executable, str(PROJECT_ROOT / 'scripts' / 'train_phase_c.py'),
         '--config', str(PROJECT_ROOT / 'configs' / 'phase_c_full_gpu.yaml')],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end='')
else:
    print('RUN_HERE = False (터미널에서 실행 권장)')
    print('학습 중에는 03_pinn_training_monitor.ipynb로 모니터링')

### 학습 중 모니터링

학습이 진행되는 동안 다른 노트북에서 모니터링:

```
notebooks/02_phase_c_development/03_pinn_training_monitor.ipynb
```

### 학습 성공 기준 (v6 Section 18)

| 항목 | 기준 | 확인 방법 |
|------|------|----------|
| BM boundary | |U| < 0.05 | Red flag check |
| Interior fringe | CoV > 0.05 | Red flag check |
| Design sensitivity | range > 0.01 | Red flag check |
| L_H convergence | < 1e-3 | Loss curve |
| No red flags | all clear | Red flag history |

### 학습 문제 시 대응

| 문제 | 증상 | 대응 |
|------|------|------|
| 자명해 수렴 | CoV → 0 | collocation 늘리기 (30K) |
| Loss 발산 | L_H → inf | lr 줄이기 (1e-4) |
| BM 미학습 | BM |U| > 0.05 | lambda_BC 높이기 (1.0) |
| 느린 수렴 | Loss plateau | warm start 먼저 해보기 |

In [ ]:
# ── 학습 완료 확인 ──
import torch

final_ckpt = PROJECT_ROOT / 'checkpoints' / 'phase_c_final.pt'
if final_ckpt.exists():
    ckpt = torch.load(final_ckpt, map_location='cpu', weights_only=False)
    print(f'phase_c_final.pt: epoch {ckpt["epoch"]}')
    
    # Quick eval
    from backend.training.red_flag_detector import detect_red_flags
    from backend.core.pinn_model import PurePINN
    cfg = ckpt.get('config', {'hidden_dim': 128, 'num_layers': 4, 'num_freqs': 48, 'omega_0': 30.0})
    model = PurePINN(**cfg)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    
    report = detect_red_flags(model, torch.device('cpu'))
    print(f'CoV: {report.interior_cov:.4f} (want > 0.05)')
    print(f'BM1: {report.bm1_mean_amp:.6f} (want < 0.05)')
    print(f'Sensitivity: {report.design_sensitivity:.6f} (want > 0.01)')
    print(report.summary())
    
    if not report.has_red_flag and not report.has_warning:
        print('\n>>> Step 1 PASS! Step 2로 진행하세요.')
    else:
        print('\n>>> 추가 학습 또는 파라미터 조정 필요')
else:
    print('phase_c_final.pt 없음. Step 1 학습을 먼저 실행하세요.')

---
# Step 2: LightTools 시뮬레이션 (★ Step 1과 동시 진행 가능)

## TMM+ASM 입사광을 LT에 직접 주입하여 시뮬

### 2a. Source 파일 생성 (이미 완료됨)

In [ ]:
# ── LT Source 파일 확인 ──
source_dir = PROJECT_ROOT / 'data' / 'lt_source_files'
source_files = sorted(source_dir.glob('source_theta*.txt')) if source_dir.exists() else []

if source_files:
    print(f'LT Source files: {len(source_files)} files')
    for f in source_files:
        print(f'  {f.name} ({f.stat().st_size/1024:.0f} KB)')
    print()
    print('이 파일들을 LightTools Custom Source로 import하세요.')
    print('상세 방법: notebooks/02_phase_c_development/07_lighttools_integration.ipynb')
else:
    print('Source 파일 없음. 생성합니다...')
    from backend.physics.asm_lut_generator import export_lt_source_files
    files = export_lt_source_files(
        lut_path=str(PROJECT_ROOT / 'data' / 'asm_luts' / 'incident_z40.npz'),
        output_dir=str(source_dir),
        theta_subset=[0, 10, 15, 20, 25, 30, 35, 40],
    )
    print(f'Generated {len(files)} files')

### 2b. LightTools 실행 가이드

```
LightTools 모델 구성 (간소화 버전 — AR+CG 불필요!):

  ┌─ Custom Source (z=40) ← data/lt_source_files/source_theta+XX.X.txt
  ├─ BM2 Aperture (z=40, w2, delta2)
  ├─ ILD (z=20~40, n=1.52)
  ├─ BM1 Aperture (z=20, w1, delta1)
  ├─ Encap (z=0~20, n=1.52)
  └─ OPD Receiver (z=0, 504um, 1000px)

각 시뮬마다:
  1. Source 파일 교체 (theta별)
  2. BM1 w1, delta1 변경
  3. BM2 w2, delta2 변경
  4. 시뮬 실행
  5. OPD Receiver 데이터 추출 → sim_XXXX.npz 저장
```

In [ ]:
# ── 시뮬 계획 확인/생성 ──
plan_path = PROJECT_ROOT / 'data' / 'lt_checkpoint' / 'simulation_plan.json'

if plan_path.exists():
    import json
    with open(plan_path) as f:
        plan = json.load(f)
    print(f'Simulation plan: {plan["n_total"]} runs')
    print(f'Designs: {plan["n_designs"]}, Angles: {plan["n_angles"]}')
    print(f'Theta values: {plan["theta_values"]}')
    print()
    print('First 5 configs:')
    for cfg in plan['configs'][:5]:
        print(f'  #{cfg["sim_id"]:3d}: d1={cfg["delta_bm1"]:+6.1f} d2={cfg["delta_bm2"]:+6.1f} '
              f'w1={cfg["w1"]:5.1f} w2={cfg["w2"]:5.1f} th={cfg["theta_deg"]:4.0f}')
else:
    print('Plan 없음. 생성합니다...')
    from backend.data.lhs_sampler import generate_lhs_samples, save_simulation_plan
    plan_data = generate_lhs_samples(n_samples=40, n_angles=5, seed=42)
    save_simulation_plan(plan_data, str(plan_path))
    print(f'Generated: {plan_data["n_total"]} simulations')

In [ ]:
# ── LT 결과 저장 포맷 (수동 실행 시 참고) ──
print('LT 결과 저장 코드 (각 시뮬 후):')
print()
print('import numpy as np')
print()
print('# LT에서 추출한 OPD Receiver 데이터')
print('x_array = ...       # (N,) x 좌표 (um)')
print('I_array = ...       # (N,) intensity')
print()
print('# PSF 7 pixel 계산')
print('psf_7 = np.zeros(7)')
print('for i in range(7):')
print('    center = i * 72 + 36')
print('    mask = (x_array >= center - 5) & (x_array <= center + 5)')
print('    dx = x_array[1] - x_array[0]')
print('    psf_7[i] = I_array[mask].sum() * dx')
print()
print('np.savez(f"data/lt_results/sim_{sim_id:04d}.npz",')
print('    intensity=I_array, x_coords=x_array, psf_7=psf_7,')
print('    delta_bm1=d1, delta_bm2=d2, w1=w1, w2=w2, theta_deg=theta)')

In [ ]:
# ── LT 결과 확인 ──
lt_dir = PROJECT_ROOT / 'data' / 'lt_results'
lt_files = sorted(lt_dir.glob('sim_*.npz')) if lt_dir.exists() else []

print(f'LT results: {len(lt_files)} files')
if lt_files:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, min(4, len(lt_files)), figsize=(4*min(4, len(lt_files)), 4))
    if not isinstance(axes, (list, np.ndarray)): axes = [axes]
    for ax, f in zip(axes, lt_files[:4]):
        data = np.load(f)
        ax.plot(data['x_coords'], data['intensity'], 'b-', lw=0.8)
        ax.set_title(f.stem, fontsize=8)
        ax.grid(True, alpha=0.2)
    plt.suptitle('LightTools Results', fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f'\n>>> Step 2 완료! LT 데이터 {len(lt_files)}개 확보.')
else:
    print('아직 LT 결과 없음.')
    print('Step 2를 완료한 후 이 셀을 다시 실행하세요.')
    print()
    print('★ Step 1 (PINN 학습)과 동시에 진행 가능합니다!')

---
# Step 3: L_I Fine-tuning (LT 데이터 반영)

Step 1의 PINN에 LT 데이터(L_I)를 추가하여 fine-tuning합니다.

```
Before: L_H + L_phase + L_BC (물리만)
After:  L_H + L_phase + L_BC + L_I (물리 + 데이터)
                                 ↑
                          LT 시뮬 강도 데이터
```

In [ ]:
# ── L_I Fine-tuning 실행 ──
has_pinn = (PROJECT_ROOT / 'checkpoints' / 'phase_c_final.pt').exists()
has_lt = len(lt_files) > 0

print('L_I Fine-tuning 준비 상태:')
print(f'  PINN checkpoint: {"OK" if has_pinn else "MISSING (Step 1 필요)"}')
print(f'  LT results:     {len(lt_files)} files {"OK" if has_lt else "(Step 2 필요)"}')
print()

if has_pinn and has_lt:
    print('=== Fine-tuning 실행 명령어 ===')
    print()
    print('python scripts/train_phase_c.py \\')
    print('    --resume checkpoints/phase_c_final.pt \\')
    print('    --epochs 55000 \\')
    print('    --no-warmstart \\')
    print('    --tag l_i_finetune')
    print()
    print('configs/phase_c_full_gpu.yaml에서 lambda_I 변경 필요:')
    print('  curriculum:')
    print('    lambda_I: 0.3    # 0.0 → 0.3')
elif not has_pinn:
    print('→ Step 1 (PINN 학습)을 먼저 완료하세요.')
else:
    print('→ Step 2 (LT 시뮬)를 먼저 완료하세요.')
    print('  또는 L_I 없이 Step 4로 바로 진행 가능 (Phase C-1)')

---
# Step 4: FNO 재증류

GPU 학습된 PINN에서 FNO surrogate를 재학습합니다.

In [ ]:
print('=== FNO 증류 명령어 ===')
print()
print('# GPU에서 실행 (10분)')
print('python scripts/distill_fno.py --epochs 2000 --device cuda')
print()
print('# CPU에서 실행 (1시간)')
print('python scripts/distill_fno.py --epochs 1000 --device cpu')
print()

fno_ckpt = PROJECT_ROOT / 'checkpoints' / 'fno_surrogate.pt'
if fno_ckpt.exists():
    ckpt = torch.load(fno_ckpt, map_location='cpu', weights_only=False)
    print(f'FNO checkpoint exists:')
    print(f'  Val loss: {ckpt["best_val_loss"]:.6f}')
    print(f'\n>>> GPU PINN으로 재증류하면 정확도 향상됩니다.')
else:
    print('FNO checkpoint 없음. 증류를 실행하세요.')

---
# Step 5: BoTorch 역설계

FNO surrogate를 사용하여 최적 BM 설계를 찾습니다.

In [ ]:
if fno_ckpt.exists():
    from backend.core.botorch_optimizer import run_inverse_design
    
    print('Running inverse design...')
    result = run_inverse_design(
        fno_checkpoint=str(fno_ckpt),
        n_initial=30,
        n_iterations=30,
        batch_size=4,
    )
    
    print(f'\nBest design:')
    for k, v in result.best_metrics.items():
        print(f'  {k}: {v:.4f}')
    print(f'\nPareto front: {len(result.pareto_params)} candidates')
    print(f'Time: {result.elapsed_sec:.0f}s')
else:
    print('FNO 없음. Step 4를 먼저 완료하세요.')

---
# Step 6: Design Studio UI

API 서버를 시작하고 브라우저에서 확인합니다.

In [ ]:
print('=== Design Studio 실행 ===')
print()
print('# 1. API 서버 시작')
print('python -m uvicorn backend.api.main:app --port 8000')
print()
print('# 2. 브라우저에서 열기')
print(f'file:///{PROJECT_ROOT}/frontend/index.html')
print()
print('# 3. API 직접 테스트')
print('curl http://localhost:8000/api/health')
print()
print('UI 기능:')
print('  Tab 1: PSF Inference  - 슬라이더로 설계변수 조절 → PSF 예측')
print('  Tab 2: Inverse Design - 목표 MTF 입력 → BoTorch 역설계')
print('  Tab 3: System Status  - PINN/FNO 로딩 상태')

---
# Final Evaluation

In [ ]:
print('=== 최종 평가 실행 ===')
print()
print('python scripts/evaluate_phase_c.py --checkpoint checkpoints/phase_c_final.pt')
print()
print('또는 노트북:')
print('  notebooks/02_phase_c_development/04_pinn_evaluation.ipynb')
print('  notebooks/02_phase_c_development/06_phase_c_report.ipynb')

---
# Checklist

| # | 단계 | 소요 시간 | 완료 |
|---|------|----------|:----:|
| 0 | GPU 환경 사전 점검 | 1분 | [ ] |
| 0a | ASM LUT 생성 (없으면) | 1분 | [ ] |
| 1 | PINN GPU 학습 (20K colloc, 50K ep) | 4-8시간 | [ ] |
| 1a | Red flag 체크 통과 | - | [ ] |
| 2 | LT Source 파일 → LightTools | 10분 | [ ] |
| 2a | LT 배치 시뮬 200 runs | 1-2일 | [ ] |
| 2b | z=40 강도 일관성 검증 | 30분 | [ ] |
| 3 | L_I Fine-tuning (lambda_I=0.3) | 2시간 | [ ] |
| 4 | FNO 재증류 | 10분 | [ ] |
| 5 | BoTorch 역설계 | 5분 | [ ] |
| 6 | Design Studio UI 확인 | 5분 | [ ] |
| 7 | 최종 평가 (evaluate_phase_c.py) | 5분 | [ ] |

### 병렬 가능 구간
```
Step 1 (GPU 학습, 4-8시간)  ←── 이 시간 동안 ──→  Step 2 (LT 시뮬)
       ↓                                              ↓
Step 3 (L_I fine-tuning, 2시간) ← 둘 다 완료 후
       ↓
Step 4-6 (빠름, 20분)
```